In [1]:
#!pip install mlflow boto3 awscli

In [4]:
#!aws configure
#!pip install dagshub
#!pip install mlflow

In [5]:
#!aws configure
import dagshub
dagshub.init(repo_owner="rohitbedse",repo_name="yt-comment-sentiment-analysis",mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=1b510838-32be-47bb-8cc1-c755fa94eb74&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e2b176434188a125ea57c4c0561578728aff9521c25109815c3d3984c0d59b48




Accessing as rohitbedse

Initialized MLflow to track repo "rohitbedse/yt-comment-sentiment-analysis"

Repository rohitbedse/yt-comment-sentiment-analysis initialized!

In [6]:
import mlflow
# Step 2: Set up the MLflow tracking server
mlflow.set_tracking_uri("https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow")

In [7]:
# Set or create an experiment
mlflow.set_experiment("Exp 4 - Handling Imbalanced Data")

2026/03/24 16:10:40 INFO mlflow.tracking.fluent: Experiment with name 'Exp 4 - Handling Imbalanced Data' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/272f5f5b8ab649eeb51b7d935d29053f', creation_time=1774368640664, experiment_id='3', last_update_time=1774368640664, lifecycle_stage='active', name='Exp 4 - Handling Imbalanced Data', tags={}, workspace='default'>

In [8]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [9]:
df = pd.read_csv('/content/reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [11]:
# Step 1: Function to run the experiment
def run_imbalanced_experiment(imbalance_method):
    ngram_range = (1, 2)  # Trigram tha ab bigram hai setting
    max_features = 1000  # Set max_features to 1000 for TF-IDF

    # Step 4: Train-test split before vectorization and resampling
    X_train, X_test, y_train, y_test = train_test_split(df['clean_comment'], df['category'], test_size=0.2, random_state=42, stratify=df['category'])

    # Step 2: Vectorization using TF-IDF, fit on training data only
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    X_train_vec = vectorizer.fit_transform(X_train)  # Fit on training data
    X_test_vec = vectorizer.transform(X_test)  # Transform test data

    # Step 3: Handle class imbalance based on the selected method (only applied to the training set)
    if imbalance_method == 'class_weights':
        # Use class_weight in Random Forest
        class_weight = 'balanced'
    else:
        class_weight = None  # Do not apply class_weight if using resampling

        # Resampling Techniques (only apply to the training set)
        if imbalance_method == 'oversampling':
            smote = SMOTE(random_state=42)
            X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)
        elif imbalance_method == 'adasyn':
            adasyn = ADASYN(random_state=42)
            X_train_vec, y_train = adasyn.fit_resample(X_train_vec, y_train)
        elif imbalance_method == 'undersampling':
            rus = RandomUnderSampler(random_state=42)
            X_train_vec, y_train = rus.fit_resample(X_train_vec, y_train)
        elif imbalance_method == 'smote_enn':
            smote_enn = SMOTEENN(random_state=42)
            X_train_vec, y_train = smote_enn.fit_resample(X_train_vec, y_train)

    # Step 5: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"Imbalance_{imbalance_method}_RandomForest_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "imbalance_handling")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, imbalance handling method={imbalance_method}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("imbalance_method", imbalance_method)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42, class_weight=class_weight)
        model.fit(X_train_vec, y_train)

        # Step 6: Make predictions and log metrics
        y_pred = model.predict(X_test_vec)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, Imbalance={imbalance_method}")
        confusion_matrix_filename = f"confusion_matrix_{imbalance_method}.png"
        plt.savefig(confusion_matrix_filename)
        mlflow.log_artifact(confusion_matrix_filename)
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_tfidf_trigrams_imbalance_{imbalance_method}")

# Step 7: Run experiments for different imbalance methods
imbalance_methods = ['class_weights', 'oversampling', 'adasyn', 'undersampling', 'smote_enn']

for method in imbalance_methods:
    run_imbalanced_experiment(method)


2026/03/24 16:23:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/24 16:24:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Imbalance_class_weights_RandomForest_TFIDF_Trigrams at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/3/runs/2d7920c9fde44cdf8f823d39d5f9086c
🧪 View experiment at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/3


2026/03/24 16:24:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/24 16:25:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Imbalance_oversampling_RandomForest_TFIDF_Trigrams at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/3/runs/14424e68e7e342dfae4afe3945540e18
🧪 View experiment at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/3


2026/03/24 16:26:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/24 16:27:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Imbalance_adasyn_RandomForest_TFIDF_Trigrams at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/3/runs/6a982657d18b40e6bda4b58a76047330
🧪 View experiment at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/3


2026/03/24 16:27:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/24 16:28:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Imbalance_undersampling_RandomForest_TFIDF_Trigrams at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/3/runs/e4a5084bf70b4c2898c5d2f47a9a6c96
🧪 View experiment at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/3


2026/03/24 16:29:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/24 16:29:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Imbalance_smote_enn_RandomForest_TFIDF_Trigrams at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/3/runs/bdfc01e3f249497695465659ab0bad16
🧪 View experiment at: https://dagshub.com/rohitbedse/yt-comment-sentiment-analysis.mlflow/#/experiments/3
